In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


df = pd.read_csv("../data/hospital_ml_ready.csv")

X = df[
    [
        'hour',
        'day_of_week',
        'is_weekend',
        'prev_hour_energy',
        'rolling_3h_avg',
        'rolling_6h_avg'
    ]
]

y = df['target_next_hour']


split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]


rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest Results:")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R² Score:", round(r2, 3))


Random Forest Results:
MAE: 27.17
RMSE: 34.23
R² Score: 0.938


In [2]:
np.random.seed(42)
dates = pd.date_range("2023-01-01", periods=8760, freq="h")

In [3]:
# Data Center
base = 1200
energy = []
for dt in dates:
    hour_factor = 1.0 + 0.05 * np.sin(2 * np.pi * dt.hour / 24)
    weekend_factor = 0.95 if dt.weekday() >= 5 else 1.0
    noise = np.random.normal(0, 30)
    energy.append(round(base * hour_factor * weekend_factor + noise, 2))


In [4]:
df_dc = pd.DataFrame({"datetime": dates, "energy_kwh": energy})
df_dc.to_csv("../data/datacenter_hourly_energy.csv", index=False)
print("Data Center done!", df_dc.head())

Data Center done!              datetime  energy_kwh
0 2023-01-01 00:00:00     1154.90
1 2023-01-01 01:00:00     1150.60
2 2023-01-01 02:00:00     1187.93
3 2023-01-01 03:00:00     1226.00
4 2023-01-01 04:00:00     1182.34


In [5]:
# MNC
np.random.seed(99)
energy = []
for dt in dates:
    if dt.weekday() >= 5:
        hour_factor = 0.3
    elif 9 <= dt.hour <= 18:
        hour_factor = 1.0 + 0.1 * np.sin(np.pi * (dt.hour - 9) / 9)
    elif 7 <= dt.hour <= 9 or 18 <= dt.hour <= 20:
        hour_factor = 0.6
    else:
        hour_factor = 0.2
    noise = np.random.normal(0, 20)
    energy.append(round(800 * hour_factor + noise, 2))

In [6]:
df_mnc = pd.DataFrame({"datetime": dates, "energy_kwh": energy})
df_mnc.to_csv("../data/mnc_hourly_energy.csv", index=False)
print("MNC done", df_mnc.head())

MNC done              datetime  energy_kwh
0 2023-01-01 00:00:00      237.15
1 2023-01-01 01:00:00      281.14
2 2023-01-01 02:00:00      245.67
3 2023-01-01 03:00:00      266.60
4 2023-01-01 04:00:00      236.91


In [7]:
for name in ["datacenter", "mnc"]:
    df = pd.read_csv(f"../data/{name}_hourly_energy.csv", parse_dates=["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    df["hour"] = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["prev_hour_energy"] = df["energy_kwh"].shift(1)
    df["rolling_3h_avg"] = df["energy_kwh"].rolling(3).mean()
    df["rolling_6h_avg"] = df["energy_kwh"].rolling(6).mean()
    df["target_next_hour"] = df["energy_kwh"].shift(-1)
    df = df.dropna().reset_index(drop=True)
    df.to_csv(f"../data/{name}_ml_ready.csv", index=False)
    print(f"{name} done Shape: {df.shape}")

datacenter done Shape: (8754, 9)
mnc done Shape: (8754, 9)


In [8]:
for name in ["datacenter", "mnc"]:
    df = pd.read_csv(f"../data/{name}_ml_ready.csv")
    
    features = ["hour", "day_of_week", "is_weekend",
                "prev_hour_energy", "rolling_3h_avg", "rolling_6h_avg"]
    X = df[features]
    y = df["target_next_hour"]
    
    split_index = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
    
    model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print("MAE:", round(mean_absolute_error(y_test, y_pred), 2))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))
    print("R² Score:", round(r2_score(y_test, y_pred), 3))
    
    joblib.dump(model, f"../model/{name}_energy_rf.pkl")
    print(f"{name} model saved")


datacenter Results:
MAE: 25.85
RMSE: 32.52
R² Score: 0.697
datacenter model saved

mnc Results:
MAE: 16.24
RMSE: 20.63
R² Score: 0.995
mnc model saved
